# Milestone 3 — Context Augmentation with RAG

A simple retrieval pipeline is used to retrieve relevant context for a question before making a prediction.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 1. Build a Small Knowledge Base

The knowledge base contains question-option information from the training data.

In [ ]:
knowledge_base = binary_train[["id", "prompt", "option", "text"]].copy()

knowledge_base = knowledge_base.drop_duplicates(
    subset=["prompt", "option"]
).reset_index(drop=True)

knowledge_base.head()

## 2. Create Document Embeddings

TF-IDF vectors are used as a simple text representation for retrieval.

In [ ]:
retriever = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    stop_words="english"
)

document_vectors = retriever.fit_transform(
    knowledge_base["text"]
)

## 3. Retrieve Relevant Context

The input question is compared with the knowledge base using cosine similarity.

In [ ]:
def retrieve_context(question, top_k=3):
    query_vector = retriever.transform([question])

    similarities = cosine_similarity(
        query_vector,
        document_vectors
    ).flatten()

    top_indices = np.argsort(similarities)[::-1][:top_k]

    return knowledge_base.iloc[top_indices].copy()

## 4. Test Retrieval

In [ ]:
question = train.iloc[0]["prompt"]

retrieved = retrieve_context(question, top_k=3)

display(retrieved)

## 5. RAG Flow

The retrieved context can be combined with the original question and answer choices before sending the input to a downstream model.

Flow:

Question → Retrieval → Relevant Context → Model → Prediction